# 5G Network Intelligence — Zero-Inflated Candidate Audit

## Hypothesis and development evidence

The mixture median reduced MAE by 0.065–0.460% in all six development fold/horizon comparisons. Test whether this small effect persists on reused validation; it does not establish performance on unseen data.


## Frozen audit procedure

For each component/horizon, take the rounded median of the two development early-stopping round counts. Refit on targets through 503, with no early stopping, probability calibration or gate tuning. Audit 504–671 on all original eligible beams. This procedure was defined after the development results and before this audit. The original final holdout stays closed; production is unchanged.


In [ ]:
RUN_TRAINING = False
SOURCE_COMMIT = '309e118e74ef1f60b498885c874bdcd0a14a6991'


## Hosted inputs and provenance

Attach completed zero-inflated development output and the small MIT refit source dataset. The source module separates fitting from audit features. No encoded source bundles or full-repository source dumps are included here.


In [ ]:
from pathlib import Path
import os, sys, shutil, subprocess, tarfile
if RUN_TRAINING:
    if not Path('/kaggle/working').is_dir():
        raise RuntimeError('Hosted Kaggle required')
    roots = [p.parent.parent for p in Path('/kaggle/input').rglob('train_hurdle_research.py') if (p.parent.parent/'artifacts/hurdle_research/manifest.json').exists()]
    if len(roots) != 1:
        raise RuntimeError('Attach one completed development output')
    PROJECT = Path('/kaggle/working/hurdle_audit')
    PROJECT.mkdir(exist_ok=True)
    for folder in ('src', 'scripts', 'configs', 'data/processed', 'artifacts/hurdle_research'):
        shutil.copytree(roots[0]/folder, PROJECT/folder, dirs_exist_ok=True, ignore=shutil.ignore_patterns('__pycache__'))
    os.chdir(PROJECT)
    sys.path.insert(0, str(PROJECT))
    os.environ['SOURCE_GIT_COMMIT'] = SOURCE_COMMIT
    Path('reports').mkdir(exist_ok=True)


## Refit implementation

One small attached module orchestrates fixed-round training. Inspect `scripts/finalize_hurdle_candidate.py` in the repository and the attached source. Shared causal features and mixture-median logic come from the completed development job.


In [ ]:
if RUN_TRAINING:
    sources = list(Path('/kaggle/input').rglob('finalize_hurdle_candidate.py'))
    if len(sources) != 1:
        raise RuntimeError('Attach exactly one refit source')
    shutil.copyfile(sources[0], 'scripts/finalize_hurdle_candidate.py')


## Fixed training and debugging

Eighteen component fits run on hosted CPU. Each component checkpoint is saved immediately. Failures retain logs and partial artifacts.


In [ ]:
if RUN_TRAINING:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'lightgbm>=4.6,<5', 'numpy', 'pandas', 'pyyaml'], check=True)
    with Path('reports/hurdle_audit_training.log').open('w') as log:
        process = subprocess.Popen([sys.executable, '-u', '-m', 'scripts.finalize_hurdle_candidate'], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        status = process.wait()
    with tarfile.open('/kaggle/working/hurdle_candidate_results.tar.gz', 'w:gz') as archive:
        for folder in ('reports', 'artifacts/hurdle_candidate'):
            if Path(folder).exists():
                archive.add(folder, arcname=folder)
    if status:
        raise RuntimeError('Refit failed; inspect saved log and checkpoints')
else:
    print('Review mode: no training or calibration.')


## Saved evaluation

MAE is in original usage units; WAPE is a ratio, not percent MAE. Compare the matched direct L1 control at all horizons. Independent checkpoint and history-only parity verification is required after download. No promotion from reused validation.


In [ ]:
paths = [Path('reports/hurdle_candidate_validation.csv'), Path('../reports/hurdle_candidate_validation.csv')]
if Path('/kaggle/input').is_dir():
    paths += list(Path('/kaggle/input').rglob('hurdle_candidate_validation.csv'))
path = next((p for p in paths if p.exists()), None)
if path:
    import pandas as pd
    from IPython.display import display
    frame = pd.read_csv(path)
    display(frame.loc[frame.load_tier == 'All', ['model', 'horizon', 'mae', 'wape', 'n_valid']])
else:
    print('No candidate audit scores available yet.')
